In [ ]:
from google.colab import drive
import tarfile
import os

drive.mount('/content/drive')

tar_path = '/content/drive/MyDrive/UD_English-EWT.tar.gz'
extract_path = '/content/data_lab5'

if not os.path.exists(extract_path):
    with tarfile.open(tar_path, 'r:gz') as tar:
        tar.extractall(path=extract_path)
    print("Giải nén thành công!")

file_list = []
for root, dirs, files in os.walk(extract_path):
    for file in files:
        if file.endswith(".conllu"):
            file_list.append(os.path.join(root, file))

train_path = [f for f in file_list if 'train' in f][0]
dev_path = [f for f in file_list if 'dev' in f][0]
print(f"File Train: {train_path}")
print(f"File Dev: {dev_path}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
File Train: /content/data_lab5/UD_English-EWT/en_ewt-ud-train.conllu
File Dev: /content/data_lab5/UD_English-EWT/en_ewt-ud-dev.conllu


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

def load_conllu(file_path):
    sentences = []
    current_sentence = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('#'):
                if not line and current_sentence:
                    sentences.append(current_sentence)
                    current_sentence = []
                continue
            parts = line.split('\t')
            if len(parts) >= 4:
                word, upos = parts[1], parts[3]
                current_sentence.append((word, upos))
    return sentences

# Load dữ liệu thực tế
train_sentences = load_conllu(train_path)
dev_sentences = load_conllu(dev_path)

# Xây dựng Vocab
word_to_ix = {"<PAD>": 0, "<UNK>": 1}
tag_to_ix = {"<PAD>": 0}

for sent in train_sentences:
    for word, tag in sent:
        if word not in word_to_ix: word_to_ix[word] = len(word_to_ix)
        if tag not in tag_to_ix: tag_to_ix[tag] = len(tag_to_ix)

ix_to_tag = {v: k for k, v in tag_to_ix.items()}
print(f"Vocab size: {len(word_to_ix)}, Tag size: {len(tag_to_ix)}")

Vocab size: 20202, Tag size: 19


In [ ]:
class POSDataset(Dataset):
    def __init__(self, sentences, word_to_ix, tag_to_ix):
        self.sentences = sentences
        self.word_to_ix = word_to_ix
        self.tag_to_ix = tag_to_ix

    def __len__(self): return len(self.sentences)

    def __getitem__(self, idx):
        sent = self.sentences[idx]
        w_idxs = torch.tensor([self.word_to_ix.get(w, self.word_to_ix["<UNK>"]) for w, t in sent])
        t_idxs = torch.tensor([self.tag_to_ix.get(t, 0) for w, t in sent])
        return w_idxs, t_idxs

def collate_fn(batch):
    sents, tags = zip(*batch)
    return pad_sequence(sents, batch_first=True, padding_value=0), \
           pad_sequence(tags, batch_first=True, padding_value=0)

train_loader = DataLoader(POSDataset(train_sentences, word_to_ix, tag_to_ix), batch_size=32, shuffle=True, collate_fn=collate_fn)
dev_loader = DataLoader(POSDataset(dev_sentences, word_to_ix, tag_to_ix), batch_size=32, collate_fn=collate_fn)

In [ ]:
class SimpleRNNForTokenClassification(nn.Module):
    def __init__(self, vocab_size, tagset_size, embed_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.rnn = nn.RNN(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, tagset_size)

    def forward(self, x):
        embeds = self.embedding(x)
        out, _ = self.rnn(embeds)
        return self.fc(out)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = SimpleRNNForTokenClassification(len(word_to_ix), len(tag_to_ix), 128, 256).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss(ignore_index=0)

def get_accuracy(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            preds = model(x).argmax(dim=-1)
            mask = (y != 0)
            correct += ((preds == y) & mask).sum().item()
            total += mask.sum().item()
    return correct / total

for epoch in range(5):
    model.train()
    total_loss = 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits.view(-1, logits.shape[-1]), y.view(-1))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    acc = get_accuracy(model, dev_loader)
    print(f"Epoch {epoch+1} | Loss: {total_loss/len(train_loader):.4f} | Dev Acc: {acc:.4f}")

Epoch 1 | Loss: 1.0265 | Dev Acc: 0.7572
Epoch 2 | Loss: 0.5951 | Dev Acc: 0.8017
Epoch 3 | Loss: 0.4463 | Dev Acc: 0.8324
Epoch 4 | Loss: 0.3484 | Dev Acc: 0.8488
Epoch 5 | Loss: 0.2772 | Dev Acc: 0.8562


In [ ]:
def predict(sentence):
    model.eval()
    words = sentence.split()
    idxs = torch.tensor([[word_to_ix.get(w, 1) for w in words]]).to(device)
    with torch.no_grad():
        p = model(idxs).argmax(dim=-1)[0]
    return [(w, ix_to_tag[p[i].item()]) for i, w in enumerate(words)]

# Test
print(predict("The head of the state is coming today"))

[('The', 'DET'), ('head', 'NOUN'), ('of', 'ADP'), ('the', 'DET'), ('state', 'NOUN'), ('is', 'AUX'), ('coming', 'VERB'), ('today', 'NOUN')]


In [ ]:
test_sentence = "I love NLP"
prediction = predict(test_sentence)
print(prediction)

[('I', 'PRON'), ('love', 'VERB'), ('NLP', 'PROPN')]
